# Pandas Part 1: Data Prep and Core Relationships

This smaller notebook covers data loading, merging, per-capita feature engineering, the Finland time-series example, global averages, and the cross-sectional GDP-vs-CO2 snapshot.

Links:
- [Pandas notebook overview](README.md)
- [Shared helpers](../functions.py)

In [ ]:
%load_ext autoreload
%autoreload 2
# Setup autoreload for shared helper edits during notebook development.

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start] + list(start.parents):
        if (candidate / 'data' / 'co2_data.csv').exists() and (candidate / 'notebooks' / 'functions.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current working directory.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
NOTEBOOKS_DIR = REPO_ROOT / 'notebooks'
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
DATA_DIR = REPO_ROOT / 'data'

In [ ]:
import pandas as pd
import numpy as np 

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay
import umap.umap_ as umap

import timeit
import requests
import json
from flatten_json import flatten

from functions import (
    apply_correlation_to_df,
    normalize_column,
    z_score_column,
    safe_divide,
    classify_income_group,
    get_top_bottom_n,
    compute_energy_mix_shares,
    plot_dual_axis_timeseries,
    run_kmeans_elbow,
)

# professional theme: clean white grid with muted palette
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.titlesize": 16,
})
%matplotlib inline

## CO2 Emissions Data

We load the Our World in Data CO2 dataset, which contains 79 columns spanning energy mix, land use, and emissions breakdowns. For this analysis we reduce it to five key variables: `country`, `year`, `iso_code`, `population`, and `co2` (total production-based CO2 emissions in millions of tonnes).

Two important filtering steps:
- **Drop rows without `iso_code`**: The dataset includes aggregate entities like "Africa", "OECD", and "World" that lack ISO country codes. Removing these ensures we work exclusively with individual nation-states.
- **Filter to post-1960**: GDP data from the World Bank only begins in 1960, so we align the time range to enable a clean merge later.

In [ ]:
co2_df = pd.read_csv(DATA_DIR / 'co2_data.csv')

# choose what columns to keep (can be changed, but this is the most important)
selected_columns = ['country', 'year', 'iso_code', 'population', 'co2']

# drop the columns that are not in the selected_columns list
co2_df = co2_df[selected_columns].copy()

# remove entries with no iso code(Continents and other groups)
co2_df = co2_df[co2_df['iso_code'].notna() & (co2_df['iso_code'].str.strip() != '')]
co2_df = co2_df[co2_df['year'] > 1960]

co2_df


## GDP Data

The World Bank GDP dataset arrives in **wide format** - one column per year (1960, 1961, ..., 2024). This is convenient for spreadsheet viewing but incompatible with tidy-data principles needed for plotting and merging. We use `pd.melt()` to reshape it into long format with one row per country-year observation.

Key steps:
- **Melt** year columns into `year` (int) and `gdp` (numeric) columns
- **Rename** `Country Code` to `iso_code` to create a shared merge key with the CO2 dataset
- **Drop missing GDP values** - not all countries have GDP records for every year, particularly in earlier decades or for newly independent states

In [ ]:
# load GDP data
gdp_df = pd.read_csv(DATA_DIR / 'gdp_data.csv')

# reshape from wide to long format
# melt the year columns into rows
year_columns = [col for col in gdp_df.columns if col.isdigit()]
gdp_df = gdp_df.melt(
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    value_vars=year_columns,
    var_name='year',
    value_name='gdp'
)

# convert year to integer and gdp to numeric
gdp_df['year'] = gdp_df['year'].astype(int)
gdp_df['gdp'] = pd.to_numeric(gdp_df['gdp'], errors='coerce')

# rename Country Code to iso_code for merging
gdp_df = gdp_df.rename(columns={'Country Code': 'iso_code'})

# keep only the columns we need
gdp_df = gdp_df[['iso_code', 'year', 'gdp']]

# remove rows with missing GDP values
gdp_df = gdp_df.dropna(subset=['gdp'])

gdp_df

## Merging the Datasets

We perform a **left join** of GDP onto the CO2 dataframe using `iso_code` and `year` as composite keys. A left join preserves every CO2 record and attaches GDP where available - countries or years without World Bank GDP data simply receive NaN. This is preferable to an inner join because it avoids silently discarding emission records that are still valuable for other analyses.

### Missing Data Heatmap

Before proceeding, we visualize data completeness. The heatmap below shows each variable as a column and each record as a row, with bright cells indicating missing values. This diagnostic is important because it reveals whether missingness is **random** or **systematic** - for instance, GDP data may be consistently absent for certain countries or time periods, which would bias any analysis that silently drops incomplete rows.

In [ ]:
# merge the datasets on iso_code and year
base_df = co2_df.merge(
    gdp_df,
    on=['iso_code', 'year'],
    how='left',
    suffixes=('_co2', '_gdp')
)

# missing data heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(base_df.isnull(), yticklabels=False, cbar=False, cmap='YlOrRd', ax=ax)
ax.set_title("Missing Data Overview After Merge")
plt.tight_layout()
plt.show()


### Data Quality Summary

The GDP column shows the most missingness - this is expected since many countries (especially newly independent or conflict-affected states) lack World Bank GDP records in earlier decades. Importantly, the missingness is **systematic** rather than random: it concentrates in specific countries and time periods. This means any analysis involving GDP will implicitly exclude these observations, so conclusions are most robust for the subset of countries with consistent economic data.

## Per-Capita Metrics

Absolute CO2 and GDP figures are dominated by population size - China and India will always top the charts simply because they have the most people, not necessarily because their economies or industries are more carbon-intensive on a per-person basis. Dividing by population yields **per-capita** values that enable fairer cross-country comparisons: how much does the average citizen emit, and how wealthy is the average citizen?

- **CO2 per capita** is expressed in **tonnes per person** (the raw CO2 column is in millions of tonnes, so we multiply by 10⁶ before dividing by population).
- **GDP per capita** is in **current USD per person**.

We use `np.divide` with a `where` guard to handle zero or missing population entries without raising division errors.

In [ ]:
# CO2 is in millions of tonnes; multiply by 1e6 to get tonnes, then divide by population
base_df['co2_per_capita'] = safe_divide(base_df['co2'].values * 1e6, base_df['population'].values)
base_df['gdp_per_capita'] = safe_divide(base_df['gdp'].values, base_df['population'].values)

# Log-transform per-capita metrics for better visualization
base_df['log_gdp_pc'] = np.log1p(base_df['gdp_per_capita'].values)
base_df['log_co2_pc'] = np.log1p(base_df['co2_per_capita'].values)

# compare raw and log-transformed data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(base_df['gdp_per_capita'].dropna(), bins=50, edgecolor='white')
axes[0].set_title('GDP per Capita – Raw Data')
axes[1].hist(base_df['log_gdp_pc'].dropna(), bins=50, edgecolor='white', color='seagreen')
axes[1].set_title('GDP per Capita – Log-Transformed (approx. normal)')
plt.tight_layout()
plt.show()


# Analysis: Distribution and transformation of the GDP-per-capita

- **Raw-Data(Left)**: The distribution leans strongly left. Majority of the countries are in the lower income groups, while a few very rich countries form a sort of tail.

- **Log-Transformation (Right)**: By using log(x) the distribution becomes closer to the Gaussian Normal Distribution. That way we can weigh relative differences of the countries equally.


# Correlation Analysis

To quantify the relationship between economic growth and carbon emissions, we compute the **Pearson correlation coefficient** (r) between GDP per capita and CO2 per capita for each country across its full available time series.

- **r close to +1** indicates strong coupling - GDP and emissions rise together, typical of industrializing economies reliant on fossil fuels.
- **r close to 0** suggests no linear relationship - the two metrics move independently.
- **r < 0** indicates **decoupling** - GDP continues to grow while emissions decline, often driven by transitions to services-based economies, renewable energy adoption, or efficiency gains.

A minimum of 5 data points per country is required to compute a meaningful correlation; countries with fewer observations are excluded. It is worth noting that Pearson r captures only *linear* association - a country that industrialized heavily, peaked, and then decoupled will show a moderate positive r despite having a clear structural break in its emission trajectory.

In [ ]:
base_df['Correlation'] = apply_correlation_to_df(base_df)

# Finland
fin = base_df[base_df['country'] == 'Finland'].dropna(subset=['gdp_per_capita', 'co2_per_capita'])

# fit a linear trend line
slope, intercept = np.polyfit(fin['gdp_per_capita'].values, fin['co2_per_capita'].values, deg=1)
print(f"Trend: CO2_pc = {slope:.6f} × GDP_pc + {intercept:.2f}")

base_df

## Country-Level Analysis: GDP vs CO2 Over Time

The dual-axis line plot below overlays GDP per capita and CO2 per capita for a selected country on a shared timeline. Two independent y-axes are necessary because the two metrics operate on vastly different scales (dollars vs. tonnes), but their *temporal trajectories* are directly comparable.

This visualization directly tests the core question of the analysis:
- **Parallel upward trends** indicate that the economy remains carbon-intensive - growth is fuelled by fossil energy.
- **Diverging trends** (GDP rising, CO2 flattening or declining) are evidence of **decoupling**, often driven by shifts toward service economies, renewable energy, or industrial efficiency.

The annotated Pearson r value provides a single-number summary of the visual relationship.

In [ ]:
# select a country for detailed time-series inspection
country = 'Finland'
df_selected = base_df[base_df['country'] == country].copy()

print(f"Selected: {country} | Years: {df_selected['year'].min()}-{df_selected['year'].max()} | "
      f"Pearson r: {df_selected['Correlation'].iloc[-1]:.3f}")
df_selected.head()

In [ ]:
plot_dual_axis_timeseries(
    df=df_selected,
    x_col='year',
    y1_col='gdp_per_capita',
    y2_col='co2_per_capita',
    y1_label='GDP per Capita (USD)',
    y2_label='CO2 per Capita (tonnes)',
    title=f'GDP per Capita vs CO2 Emissions in {country}',
    correlation_value=df_selected['Correlation'].iloc[-1]
)

### Global Averages: CO2 per Capita vs GDP per Capita Over Time

Before zooming into individual countries, we first examine the **global average** trajectories. The dual-axis line plot below overlays the mean CO2 per capita (left axis, blue) and mean GDP per capita (right axis, orange) across all countries for each year.

This bird's-eye view reveals the macro-level tension: global GDP per capita has risen steadily since 1960, while average CO2 per capita shows signs of plateauing or declining in recent decades - suggesting that **aggregate decoupling** may already be underway at the global level.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))
ax2 = ax1.twinx()

sns.lineplot(ax=ax1, x='year', y='co2_per_capita', data=base_df,
             color='steelblue', linewidth=2, label='CO2 per capita')
sns.lineplot(ax=ax2, x='year', y='gdp_per_capita', data=base_df,
             color='darkorange', linewidth=2, label='GDP per capita')

ax1.set_xlabel('Year')
ax1.set_ylabel('CO2 per capita (tonnes)', color='steelblue')
ax2.set_ylabel('GDP per capita (USD)', color='darkorange')
ax1.set_title('Global Average: CO2 per Capita vs GDP per Capita (1960-2024)')

# combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax2.get_legend().remove()

plt.tight_layout()
plt.show()

### Cross-Sectional Snapshot: Wealth vs CO2 per Capita (2024)

Time-series charts show how averages evolve, but they hide how wide the gap between countries still is at a single point in time. This log-log scatter uses the most recent year to compare countries side by side and asks a simple question: **do richer economies still emit more per person?**

To keep the picture readable, the chart filters to countries with populations above 5 million. Point size reflects population, which keeps the biggest economies visually prominent without letting microstates dominate the extremes.

In [ ]:
latest_year = int(base_df['year'].max())

# create a snapshot df based on base df
snapshot_df = (
    base_df[
        (base_df['year'] == latest_year)
        & (base_df['population'] > 5_000_000)
        & (base_df['gdp_per_capita'] > 0)
        & (base_df['co2_per_capita'] > 0)
    ][['country', 'population', 'gdp_per_capita', 'co2_per_capita']]
    .dropna()
    .copy()
)

# calculate log, trends and trendslopes
snapshot_df['pop_millions'] = snapshot_df['population'] / 1_000_000
log_gdp = np.log10(snapshot_df['gdp_per_capita'])
log_co2 = np.log10(snapshot_df['co2_per_capita'])
log_corr = np.corrcoef(log_gdp, log_co2)[0, 1]
trend_slope, trend_intercept = np.polyfit(log_gdp, log_co2, 1)
trend_x = np.linspace(log_gdp.min(), log_gdp.max(), 200)
trend_y = trend_intercept + trend_slope * trend_x

# plot the snapshot
fig, ax = plt.subplots(figsize=(11, 7))
sns.scatterplot(
    data=snapshot_df,
    x='gdp_per_capita',
    y='co2_per_capita',
    size='pop_millions',
    sizes=(40, 500),
    alpha=0.7,
    color='teal',
    edgecolor='white',
    linewidth=0.6,
    legend=False,
    ax=ax,
)
ax.plot(10 ** trend_x, 10 ** trend_y, color='darkorange', linewidth=2, linestyle='--')

# iteare through the snapshot and annotate
for _, row in snapshot_df.nlargest(6, 'population').iterrows():
    ax.annotate(
        row['country'],
        (row['gdp_per_capita'], row['co2_per_capita']),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=9,
    )

# set scales and labels
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title(f'Wealth and Emissions Still Move Together in {latest_year}')
ax.set_xlabel('GDP per Capita (USD, log scale)')
ax.set_ylabel('CO2 per Capita (tonnes, log scale)')
ax.text(
    0.03,
    0.96,
    f'Countries shown: {len(snapshot_df)}\nlog-log r = {log_corr:.2f}',
    transform=ax.transAxes,
    va='top',
)

plt.tight_layout()
plt.show()

### Interpreting the Cross-Section

The latest-year snapshot still shows a **strong positive relationship** between income and emissions: among countries above 5 million people, the log-log correlation is roughly **0.86**. In other words, richer countries still tend to emit more CO2 per person.

At the same time, the upper-right portion of the chart is far from uniform. Oil and gas exporters such as **Saudi Arabia** and the **United Arab Emirates** sit well above many other high-income economies, while countries like **Finland** and **Norway** occupy noticeably lower-emission positions for their income level. That spread is important: prosperity matters, but **energy mix and industrial structure** still explain a large part of why similarly rich countries can look very different environmentally.